# OnCallEnv Red Shift — GRPO training on hard SRE incidents

This notebook trains **Qwen-2.5-3B-Instruct** with **GRPO** (TRL + Unsloth, 4-bit LoRA on a single Kaggle T4) to act as the on-call SRE for a simulated microservice cluster, then compares it against two baselines under the same evaluator.

## What makes this run "hard"

- **Obfuscated alerts.** Every incident pages the agent with the *same* generic frontend symptom: `api-gateway` reports "High latency and elevated error rate detected in customer telemetry". The true root cause is hidden somewhere downstream (`checkout / user / payment / inventory / postgres / redis`). The agent must *trace* the fault through the dependency graph using read-only tools (`kubectl_logs`, `kubectl_top`, `jaeger_search`, `promql_query`, `dns_lookup`, ...) before it can apply the right fix.
- **Hard-task curriculum filter.** The trainer is launched with `--max-solve-rate 0.25`, which keeps only the ~73 / 206 scenarios in `curriculum_results/buffer.json` whose heuristic solve rate sits below 25%. No trivially-solvable seeds.
- **Twelve fault families × six topologies.** `oom_kill`, `cpu_hog`, `network_partition`, `dns_misconfig`, `replica_lag`, `cache_stampede`, `http_503_loop`, `deadlock`, `disk_full`, `cert_expiry`, `clock_skew`, `gc_pause` — each with a single canonical (tool, service) remediation. Topologies: `simple_fanout`, `deep_chain`, `mesh`, `star`, `bipartite`, `diamond`.

## Reward (purely behavioral, no leakage from the env)

GRPO is driven by `oncallenv.rewards.sre_shaped`, which scores **only the model's emitted text**. The legacy environment reward (which auto-credits trivially-correct RCAs and saturates around 0.9) is *not* added in. The total is clipped to `[-0.6, +1.0]` and exposed as **three disjoint reward functions** so TRL logs each as its own column:

| TRL column                                  | what it rewards                                                 |
| ------------------------------------------- | --------------------------------------------------------------- |
| `rewards/sre_format_reward/mean`            | well-formed `<thought>` + `<actions>` blocks                    |
| `rewards/sre_investigation_reward/mean`     | read-only probes on the right services *before* mutating        |
| `rewards/sre_remediation_reward/mean`       | correct (tool, service) fix + accurate `submit_rca`             |

Because the policy gradient uses the *sum* of the three, this exactly matches the original monolithic `sre_shaped_reward`.

## What we compare

1. **MLP baseline** (text features → action distribution over `(tool, service)` pairs).
2. **Qwen-2.5-3B base** (no training, prompted with the same SRE system prompt).
3. **Qwen-2.5-3B + GRPO** (this notebook's training run, evaluated via the LoRA adapter loaded over the 4-bit base with `peft`).

All three are scored by the same SRE-shaped reward on the same eval set.

## 0. Environment setup

Clone the `sre-three-way` branch onto the Kaggle worker, drop `vllm` (it conflicts with `unsloth_zoo` on Kaggle's image), and install the runtime + LLM dependencies. Training uses Unsloth; evaluation of the trained adapter goes through `transformers + peft` so we avoid pulling vLLM back in.

In [21]:
!nvidia-smi

Sat Apr 25 19:26:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
import os
from pathlib import Path

Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
os.chdir("/kaggle/working")
print("cwd:", os.getcwd())

cwd: /kaggle/working


In [23]:
import os, shutil, subprocess, time
from pathlib import Path

REPO_URL = "https://github.com/srimanreddy4/MetaHackathon-R2"
BRANCH = "sre-three-way"
WORKDIR = Path("/kaggle/working/MetaHackathon-R2")

os.chdir("/kaggle/working")
if (WORKDIR / ".git").exists():
    os.chdir(WORKDIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    if WORKDIR.exists():
        backup = WORKDIR.with_name(f"{WORKDIR.name}.bak.{int(time.time())}")
        shutil.move(str(WORKDIR), str(backup))
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-3"], check=True)

From https://github.com/srimanreddy4/MetaHackathon-R2
 * branch            sre-three-way -> FETCH_HEAD
   cf01aab..79308ee  sre-three-way -> origin/sre-three-way
Already on 'sre-three-way'


Your branch is behind 'origin/sre-three-way' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating cf01aab..79308ee
Fast-forward
 notebooks/OnCall_RedShift_GRPO_Training.ipynb | 78 +++++++++++++++------------
 scripts/train_unsloth_grpo.py                 | 30 +++++++++++
 2 files changed, 73 insertions(+), 35 deletions(-)
cwd: /kaggle/working/MetaHackathon-R2
79308ee added plots
92a529a added plots
cf01aab added medium tasks


From https://github.com/srimanreddy4/MetaHackathon-R2
 * branch            sre-three-way -> FETCH_HEAD


CompletedProcess(args=['git', 'log', '--oneline', '-3'], returncode=0)

In [24]:
%cd /kaggle/working/MetaHackathon-R2
import sys
if "src" not in sys.path:
    sys.path.append("src")
if "scripts" not in sys.path:
    sys.path.append("scripts")
print("Added src and scripts to sys.path")

/kaggle/working
Added src and scripts to sys.path


In [25]:
# Kaggle ships vllm 0.19+, which broke unsloth_zoo's `from vllm.lora.models` import.
# We don't use vllm here (training via unsloth, inference via transformers+peft), so remove it.
!pip uninstall -y -q vllm vllm-flash-attn 2>/dev/null || true
!pip install -q -U pip setuptools wheel
!pip install -q -r requirements.txt
!pip install -q -r requirements-llm.txt

## 1. Train the MLP baseline (text features only)

Reference non-LLM baseline. Trains a small MLP over text-derived features — task id one-hot, alert service one-hot, fault keyword bag-of-words, service mention bag-of-words — to pick a `(tool, service)` action. It does *not* see the privileged `ScenarioSpec` (no fault category leak), so it has to learn the same surface-level associations available to the LLM.

In [26]:
!PYTHONPATH=src:scripts python scripts/train_redshift_policy.py \
    --steps 100 \
    --batch-size 256 \
    --lr 3e-3 \
    --curriculum-buffer curriculum_results/buffer.json \
    --feature-mode text \
    --out-dir training_results/sre_mlp \
    --plots-dir docs/plots/sre_mlp \
    --plot-prefix sre_mlp

{"step": 1, "loss": 3.7266180515289307, "train_mean_reward": 0.5778377604166667, "eval_mean_reward": 0.5927040674603175}
{"step": 10, "loss": 3.1292459964752197, "train_mean_reward": 0.5607610026041667, "eval_mean_reward": 0.5716519841269841}
{"step": 20, "loss": 3.1381895542144775, "train_mean_reward": 0.5607610026041667, "eval_mean_reward": 0.5716519841269841}
{"step": 30, "loss": 3.0536746978759766, "train_mean_reward": 0.5607610026041667, "eval_mean_reward": 0.5716519841269841}
{"step": 40, "loss": 3.146547555923462, "train_mean_reward": 0.5737754557291667, "eval_mean_reward": 0.5716519841269841}
{"step": 50, "loss": 3.039031982421875, "train_mean_reward": 0.5737754557291667, "eval_mean_reward": 0.5716519841269841}
{"step": 60, "loss": 3.0388948917388916, "train_mean_reward": 0.5737754557291667, "eval_mean_reward": 0.5716519841269841}
{"step": 70, "loss": 3.0448336601257324, "train_mean_reward": 0.5737754557291667, "eval_mean_reward": 0.5716519841269841}
{"step": 80, "loss": 2.9928

## 2. GRPO-train Qwen-2.5-3B with the three-way SRE reward

Calls `scripts/train_unsloth_grpo.py` with `--reward-mode sre --prompt-mode sre`, which:

- builds the SRE system prompt (`oncallenv.rewards.sre_shaped.SRE_SYSTEM_PROMPT`) demanding `<thought>` + `<actions>` + `submit_rca`;
- filters the regret buffer down to genuinely hard scenarios via `--max-solve-rate 0.25`;
- registers the three split reward functions (`sre_format_reward`, `sre_investigation_reward`, `sre_remediation_reward`) so TRL logs each as its own column and the policy gradient is taken over their sum;
- LoRA-trains over `q/k/v/o/gate/up/down` projections (rank 16, alpha 32) on a 4-bit base.

The summary JSON written to `training_results/sre_qwen_grpo/` reports the baseline-vs-trained mean reward and the per-task generations for inspection.

In [27]:
!PYTHONPATH=src:scripts python scripts/train_unsloth_grpo.py \
    --model-name unsloth/Qwen2.5-3B-Instruct-bnb-4bit \
    --out-dir training_results/sre_qwen_grpo \
    --curriculum-buffer curriculum_results/buffer.json \
    --reward-mode sre \
    --prompt-mode sre \
    --max-tasks 120 \
    --max-steps 200 \
    --save-steps 20 \
    --eval-tasks 24 \
    --num-generations 4 \
    --max-completion-length 256 \
    --max-prompt-length 1024 \
    --max-seq-length 1536 \
    --lr 5e-6 \
    --temperature 0.8 \
    --logging-steps 1

[curriculum] keeping 54/206 medium-difficulty scenarios (0.40 <= solve_rate <= 0.70)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 19:27:35.025865: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777145255.049175    2059 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777145255.056674    2059 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777145255.076289    2059 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777145255.076317    2059 computation_placer.cc:177] computation placer already registered. 

In [ ]:
!ls training_results/sre_qwen_grpo/summary.json

training_results/sre_qwen_grpo/summary.json


In [ ]:
!cat training_results/sre_qwen_grpo/trained_generations.json

{
  "mean_reward": -0.16874999999999998,
  "scores": {
    "evolved_0040_db34385e57": 0.15,
    "evolved_0195_6ad30444d4": -0.19999999999999996,
    "evolved_0165_b13c686c48": 0.15000000000000005,
    "evolved_0099_19a4df519f": -0.35,
    "evolved_0031_a03b9ffd00": 0.25,
    "evolved_0062_5873fbe749": -0.44999999999999996,
    "evolved_0109_61eca74ce9": 0.15,
    "evolved_0034_af7905bff6": -0.35,
    "evolved_0135_66da916196": -0.05000000000000002,
    "evolved_0089_3526439777": -0.44999999999999996,
    "evolved_0154_1877418d50": -0.35,
    "evolved_0136_4546b58c65": 5.551115123125783e-17,
    "evolved_0001_823796c06b": -0.24999999999999994,
    "evolved_0027_45da8c779b": -0.44999999999999996,
    "evolved_0132_8229eb045a": -0.44999999999999996,
    "evolved_0115_c4cc51f8bf": -0.44999999999999996,
    "evolved_0049_757d8279d3": 0.29999999999999993,
    "evolved_0183_468a9382d9": -0.35,
    "evolved_0173_fc325576d4": -0.35,
    "evolved_0182_41772cd926": -0.44999999999999996,
    "evol

## 3. Universal SRE evaluator

All three models go through the same scorer: `sre_shaped_reward(text, task_id)`.

- **Qwen models** receive the SRE system prompt and emit `<thought>` + `<actions>` + `submit_rca` directly.
- **MLP** can only emit raw `(tool, service)` picks, so we wrap its top-3 actions in a fake `<actions>` block before scoring. This means the MLP forfeits all `<thought>` and RCA bonuses by construction; it can only earn investigation + remediation points if its picks happen to land on the right service.

In [ ]:
import sys
if "src" not in sys.path:
    sys.path.append("src")
if "scripts" not in sys.path:
    sys.path.append("scripts")

import json, random
from pathlib import Path
import numpy as np
import torch
from tqdm import tqdm

from oncallenv.rewards.sre_shaped import (
    SRE_SYSTEM_PROMPT,
    build_sre_prompt,
    get_ground_truth,
    sre_shaped_reward,
)
from oncallenv.core.env import OnCallRedShiftEnv
from scripts.train_redshift_policy import (
    ACTIONS,
    build_policy,
    feature_vector,
    inspect_task,
)

# Same eval pool as notebook 06 for an apples-to-apples story.
buf = json.loads(Path("curriculum_results/buffer.json").read_text())
scenarios = buf if isinstance(buf, list) else buf.get("scenarios", [])
all_task_ids = list(dict.fromkeys([s["spec"]["task_id"] for s in scenarios]))
random.seed(42)
eval_tasks = random.sample(all_task_ids, 40)
task_truth = {tid: get_ground_truth(tid) for tid in eval_tasks}
print(f"Evaluating {len(eval_tasks)} tasks under SRE-shaped reward.")

Evaluating 40 tasks under SRE-shaped reward.


In [ ]:
def sre_evaluate(text_fn, task_ids, label):
    breakdown_keys = [
        "env_reward",
        "thought_format_bonus", "thought_substance_bonus", "thought_semantic_bonus",
        "actions_format_bonus", "first_cmd_bonus", "read_only_first_bonus",
        "first_targets_root_bonus", "read_before_mutate_bonus",
        "correct_remediation_bonus", "declare_resolved_bonus",
        "rca_emitted_bonus", "rca_service_bonus", "rca_category_bonus",
        "no_actions_penalty", "no_commands_penalty", "no_rca_penalty",
        "truncation_penalty", "verbose_penalty", "shotgun_penalty",
        "early_mutate_penalty", "wrong_category_penalty",
    ]
    totals, accum = [], {k: 0.0 for k in breakdown_keys}
    for tid in tqdm(task_ids, desc=f"Eval {label}"):
        text = text_fn(tid)
        score = sre_shaped_reward(text, tid, truth=task_truth[tid], return_breakdown=True)
        totals.append(score.total)
        d = score.as_dict()
        for k in breakdown_keys:
            accum[k] += d[k]
    n = max(1, len(task_ids))
    summary = {"label": label, "mean_total": float(np.mean(totals))}
    summary.update({k: accum[k] / n for k in breakdown_keys})
    print(f"{label} mean SRE reward: {summary['mean_total']:.4f}")
    return summary, totals

## 4. Evaluate the MLP under the SRE-shaped reward

Wrap the MLP's top-3 action picks in an `<actions>` block and score. Without a `<thought>` it loses up to +0.20 in format bonuses; without `submit_rca` it eats the −0.15 `no_rca_penalty`. Its only path to a positive total is correctly mutating the right service.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dummy_info = inspect_task(eval_tasks[0])
input_dim = len(feature_vector(dummy_info, feature_mode="text"))
mlp_policy = build_policy(input_dim, torch.nn).to(device)
mlp_path = Path("training_results/sre_mlp/policy.pt")
if mlp_path.exists():
    mlp_policy.load_state_dict(torch.load(mlp_path, map_location=device))
mlp_policy.eval()

def mlp_text(task_id):
    info = inspect_task(task_id)
    x = torch.tensor([feature_vector(info, feature_mode="text")], dtype=torch.float32, device=device)
    with torch.no_grad():
        logits = mlp_policy(x)[0]
    chosen = torch.topk(logits, k=3).indices.tolist()
    cmds = [f"{ACTIONS[i][0]} {ACTIONS[i][1]}" for i in chosen]
    body = "\n".join(cmds + ["declare_resolved"])
    return f"<actions>\n{body}\n</actions>"

mlp_summary, mlp_totals = sre_evaluate(mlp_text, eval_tasks, "MLP (Text Mode)")

Eval MLP (Text Mode): 100%|██████████| 40/40 [00:00<00:00, 144.72it/s]

MLP (Text Mode) mean SRE reward: 0.0950


## 5. Evaluate Qwen-2.5-3B (untrained) under the SRE-shaped reward

Same prompt the GRPO run was trained on, but using the raw 4-bit base. We bypass Unsloth/vLLM here and load the model with `transformers + bitsandbytes` so the adapter-load step in §6 stays compatible with Kaggle's image.

In [ ]:
# Bypass unsloth/vLLM on Kaggle by loading the 4-bit base model with transformers+bitsandbytes
# and applying the GRPO LoRA adapter (when present) via peft. This avoids the
# `unsloth_zoo -> vllm.lora.models` import chain that breaks on Kaggle's vLLM build.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
qwen_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
qwen_base.eval()

def qwen_text_factory(model, label):
    def _generate(task_id):
        env = OnCallRedShiftEnv()
        obs = env.reset(task_id=task_id)
        prompt = build_sre_prompt(
            task_id=task_id,
            alert_service=obs.alerts[0].service,
            alert_message=obs.alerts[0].message,
            services=list(obs.services),
        )
        messages = [
            {"role": "system", "content": SRE_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(chat, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
            )
        return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    _generate.__name__ = f"qwen_{label}"
    return _generate

qwen_base_summary, qwen_base_totals = sre_evaluate(qwen_text_factory(qwen_base, "base"), eval_tasks, "Qwen 3B (Base)")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!
2026-04-25 18:38:32.961150: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777142312.986198     102 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777142312.991999     102 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777142313.007652     102 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777142313.007666     102 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777142313.007668     102

model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Eval Qwen 3B (Base):  28%|██▊       | 11/40 [06:05<16:04, 33.25s/it]


KeyboardInterrupt: 

## 6. Evaluate Qwen-2.5-3B + GRPO under the SRE-shaped reward

Free the base-model GPU memory, then re-load the same 4-bit base and stack the GRPO LoRA adapter on top with `peft.PeftModel`. Rerun the eval set through the identical decoding setup as §5 so any delta is attributable to the adapter.

In [ ]:
import gc
from peft import PeftModel

# Free the base model before loading the GRPO copy to keep the T4 happy.
try:
    del qwen_base
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

adapter_dir = Path("training_results/sre_qwen_grpo/adapter")
if not adapter_dir.exists():
    # Fall back to the latest checkpoint adapter if the merged adapter dir wasn't saved
    ckpts = sorted(Path("training_results/sre_qwen_grpo").glob("checkpoint-*"),
                   key=lambda p: int(p.name.split("-")[-1]))
    assert ckpts, f"No adapter dir or checkpoint found under training_results/sre_qwen_grpo"
    adapter_dir = ckpts[-1]
    print(f"Using checkpoint adapter: {adapter_dir}")

qwen_grpo_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
qwen_grpo = PeftModel.from_pretrained(qwen_grpo_base, str(adapter_dir))
qwen_grpo.eval()

qwen_grpo_summary, qwen_grpo_totals = sre_evaluate(qwen_text_factory(qwen_grpo, "grpo"), eval_tasks, "Qwen 3B (GRPO+SRE)")

## 7. Three-way comparison and reward breakdown

Two plots:

1. **Mean SRE-shaped total reward** across MLP / Qwen-base / Qwen-GRPO on the held-out eval set.
2. **Stacked decomposition** of the per-component bonuses and penalties (`thought_*`, `actions_format`, `first_targets_root`, `read_before_mutate`, `correct_remediation`, `rca_service`, `rca_category`, `shotgun`, `no_rca`, ...). This is what makes the gain from GRPO interpretable: you can see whether the model improved on format, on investigation, or on remediation correctness.

In [ ]:
import matplotlib.pyplot as plt

summaries = [mlp_summary, qwen_base_summary, qwen_grpo_summary]
labels = [s["label"] for s in summaries]
totals = [s["mean_total"] for s in summaries]
colors = ["#16a34a", "#2563eb", "#dc2626"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, totals, color=colors, edgecolor="white", linewidth=1.5)
ax.set_ylabel("Mean SRE-shaped reward (40 tasks)")
ax.set_title("SRE-Shaped Reward: GRPO + thought + targeted fix wins")
ax.grid(axis="y", alpha=0.3)
for bar, score in zip(bars, totals):
    ax.text(bar.get_x() + bar.get_width() / 2.0, score + 0.02, f"{score:.3f}", ha="center", fontsize=12, fontweight="bold")
Path("docs/plots/sre_three_way").mkdir(parents=True, exist_ok=True)
fig.savefig("docs/plots/sre_three_way/sre_three_way.png", dpi=160)
plt.show()

In [ ]:
components = [
    ("thought_format_bonus", "#0ea5e9"),
    ("thought_substance_bonus", "#0284c7"),
    ("thought_semantic_bonus", "#22d3ee"),
    ("actions_format_bonus", "#14b8a6"),
    ("first_cmd_bonus", "#34d399"),
    ("read_only_first_bonus", "#10b981"),
    ("first_targets_root_bonus", "#059669"),
    ("read_before_mutate_bonus", "#16a34a"),
    ("correct_remediation_bonus", "#65a30d"),
    ("declare_resolved_bonus", "#84cc16"),
    ("rca_emitted_bonus", "#a3a3a3"),
    ("rca_service_bonus", "#a855f7"),
    ("rca_category_bonus", "#f59e0b"),
    ("no_actions_penalty", "#dc2626"),
    ("no_commands_penalty", "#b91c1c"),
    ("no_rca_penalty", "#991b1b"),
    ("truncation_penalty", "#7f1d1d"),
    ("verbose_penalty", "#fb923c"),
    ("shotgun_penalty", "#ef4444"),
    ("early_mutate_penalty", "#f97316"),
    ("wrong_category_penalty", "#c2410c"),
]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(summaries))
pos = np.zeros(len(summaries))
neg = np.zeros(len(summaries))
for key, color in components:
    vals = np.array([s[key] for s in summaries])
    pos_mask = vals >= 0
    if pos_mask.any():
        ax.bar(x, np.where(pos_mask, vals, 0), bottom=pos, color=color, edgecolor="white", label=key)
        pos = pos + np.where(pos_mask, vals, 0)
    neg_mask = vals < 0
    if neg_mask.any():
        ax.bar(x, np.where(neg_mask, vals, 0), bottom=neg, color=color, edgecolor="white", label=key if not pos_mask.any() else None)
        neg = neg + np.where(neg_mask, vals, 0)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Mean reward contribution")
ax.set_title("Why GRPO wins: per-component reward breakdown")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("docs/plots/sre_three_way/sre_breakdown.png", dpi=160, bbox_inches="tight")
plt.show()

out = {"mlp": mlp_summary, "qwen_base": qwen_base_summary, "qwen_grpo": qwen_grpo_summary}
Path("docs/plots/sre_three_way/sre_three_way.json").write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))